# Evaluation figures recomputed from `raw_dataset/`

Every figure below is derived on the fly from the files under `../raw_dataset/`; nothing is read
from the paper. Sections follow the paper's evaluation: operational overview, threat categories,
cross-platform monitoring, same-input baseline comparison (novelty, coverage, VirusTotal, Tranco),
public-feed overlap and early detection, ablation study, and manual review.

Inputs (all under `raw_dataset/`): `reports.jsonl` and `evaluation_evidence/*`.

In [ ]:
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

RAW = Path('..') / 'raw_dataset'
EV = RAW / 'evaluation_evidence'
def rows(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]
reports = rows(RAW / 'reports.jsonl')
direct = rows(EV / 'same_input_direct_records.jsonl')
baselines = rows(EV / 'baseline_ioc_records.jsonl')
feeds = rows(EV / 'public_feed_overlap_records.jsonl')
timing = rows(EV / 'public_feed_timing_records.jsonl')
ablation = rows(EV / 'ablation_membership.jsonl')
ablation_params = json.loads((EV / 'ablation_parameters.json').read_text(encoding='utf-8'))
manual = rows(EV / 'manual_review_records.jsonl')
overview = json.loads((EV / 'pipeline_overview.json').read_text(encoding='utf-8'))
PLATS = ['x', 'reddit', 'telegram', 'discord']
CATS = ['Vulnerability', 'Phishing/Fraud', 'APT/Ransomware', 'Malware/C2']
FEEDS = ['PhishTank', 'URLhaus', 'MalwareBazaar', 'CISA KEV', 'TweetFeed', 'OTX Pulses']
results = {}
pct = lambda a, b: round(100 * a / b, 1) if b else None
print(len(reports), 'reports')

## 1. Operational overview (31-day deployment)

In [ ]:
raw = overview['raw_posts_by_platform']; total = overview['raw_posts_total']
print(f"Raw posts: {total:,}")
for p, n in raw.items():
    print(f"  {p:<10} {n:>8,} ({pct(n, total)}%)")
print(f"Unique clusters: {overview['unique_clusters']:,}")
print(f"  LOW-severity clusters filtered at Step 2: {overview['step2_low_severity_filtered_clusters']:,}")
print(f"  Clusters skipped at Step 4: {overview['step4_skip_clusters']:,}")
print(f"  Actionable reports: {overview['actionable_reports']:,} (= {overview['unique_clusters']:,} - {overview['step2_low_severity_filtered_clusters']:,} - {overview['step4_skip_clusters']:,})")
assert overview['unique_clusters'] - overview['step2_low_severity_filtered_clusters'] - overview['step4_skip_clusters'] == len(reports)
results['overview'] = overview

## 2. Threat categories, severity, cross-platform reports, IoCs per report

In [ ]:
cats = Counter(r['paper_category'] for r in reports); sev = Counter(r['severity'] for r in reports)
multi = sum(len(r['platforms']) > 1 for r in reports); total_iocs = sum(r['iocs']['count'] for r in reports)
results['reports'] = {'reports': len(reports), 'categories': {c: cats[c] for c in CATS}, 'severity': dict(sev), 'cross_platform_reports': multi, 'cross_platform_pct': pct(multi, len(reports)), 'ioc_total': total_iocs, 'mean_iocs_per_report': round(total_iocs / len(reports), 2)}
for c in CATS:
    print(f"  {c:<15} {cats[c]:>5,} ({pct(cats[c], len(reports))}%)")
print('Severity:', {k: sev[k] for k in ['CRITICAL', 'HIGH', 'MEDIUM'] if sev[k]})
print(f"Cross-platform reports: {multi:,} ({pct(multi, len(reports))}%)   IoCs: {total_iocs:,} ({total_iocs/len(reports):.2f} per report)")

## 3. Platform contribution per category and single-platform removal loss

In [ ]:
table = {}
print(f"{'Category':<16}" + ''.join(f"{p:>10}" for p in PLATS) + f"{'Cross-plt.':>12}")
for cat in CATS:
    cr = [r for r in reports if r['paper_category'] == cat]
    row = {pl: pct(sum(pl in r['platforms'] for r in cr), len(cr)) for pl in PLATS}
    row['cross_platform'] = pct(sum(len(r['platforms']) > 1 for r in cr), len(cr))
    row['single_platform_only'] = {pl: pct(sum(r['platforms'] == [pl] for r in cr), len(cr)) for pl in PLATS}
    table[cat] = row
    print(f"{cat:<16}" + ''.join(f"{row[p]:>9.1f}%" for p in PLATS) + f"{row['cross_platform']:>11.1f}%")
print('\nReports lost without the platform (single-platform reports, % of category):')
for cat in CATS:
    print(f"  {cat:<16}" + ''.join(f"{p}: {table[cat]['single_platform_only'][p]:>5.1f}%  " for p in PLATS))
results['platform_contribution'] = table

## 4. Same-input IoC comparison with baselines (Phishing/Fraud and Malware/C2)

TIBlender (D) rows come from `same_input_direct_records.jsonl`; baseline rows from
`baseline_ioc_records.jsonl`. Novelty = share of TIBlender (D) IoCs of a category that the baseline
did not extract; Coverage = share of the baseline's IoCs that TIBlender (D) also extracted.

In [ ]:
def bucket(n):
    return '0' if n == 0 else '1-5' if n <= 5 else '6-10' if n <= 10 else '11+'
def quality(items):
    n = len(items); bins = Counter(bucket(x['vt_malicious']) for x in items)
    domains = [x for x in items if x['type'] == 'domain']
    return {'total': n, 'vt_pct': {b: pct(bins[b], n) for b in ['0', '1-5', '6-10', '11+']}, 'tranco_pct': pct(sum(bool(x.get('tranco_top1m')) for x in domains), len(domains))}
tib = defaultdict(list)
for x in direct:
    tib[x['platform']].append(x)
base = defaultdict(list)
for x in baselines:
    base[(x['system'], x['platform'])].append(x)
results['same_input'] = []
print(f"{'Platform':<9}{'System':<12}{'Total':>6}{'Ph':>5}{'Mal':>5}{'Nov.Ph':>8}{'Nov.Mal':>8}{'Cov.':>7}   VT 0    1-5   6-10    11+  Tranco")
for pl in PLATS:
    tset = {(x['type'], x['value']) for x in tib[pl]}
    tcat = Counter(x['category'] for x in tib[pl])
    systems = [('TIBlender (D)', tib[pl])] + [(s, base[(s, pl)]) for s in ['Tweezers', 'DarkGram', 'Reddit2CTI', 'iocsearcher'] if (s, pl) in base]
    for name, items in systems:
        q = quality(items); c = Counter(x['category'] for x in items)
        row = {'platform': pl, 'system': name, 'total': q['total'], 'phishing_fraud': c['Phishing/Fraud'], 'malware_c2': c['Malware/C2'], 'vt_pct': q['vt_pct'], 'tranco_pct': q['tranco_pct']}
        if name == 'TIBlender (D)':
            nov = {}
            for s, bitems in systems[1:]:
                bset = {(x['type'], x['value']) for x in bitems}
                nov[s] = {cat: pct(sum((x['type'], x['value']) not in bset for x in tib[pl] if x['category'] == cat), tcat[cat]) for cat in ['Phishing/Fraud', 'Malware/C2'] if tcat[cat]}
            row['novelty_vs_baseline'] = nov
            row['coverage_of_baseline'] = None
            novtxt = ' '.join(f"{s[:4]}:" + '/'.join(f"{v}" for v in n.values()) for s, n in nov.items())
        else:
            bset = {(x['type'], x['value']) for x in items}
            row['coverage_of_baseline'] = pct(len(bset & tset), len(bset))
            novtxt = ''
        results['same_input'].append(row)
        cov = f"{row['coverage_of_baseline']:.1f}%" if row['coverage_of_baseline'] is not None else '   -'
        print(f"{pl:<9}{name:<12}{q['total']:>6}{c['Phishing/Fraud']:>5}{c['Malware/C2']:>5}{'':>16}{cov:>7}   " + ' '.join(f"{q['vt_pct'][b]:5.1f}%" for b in ['0', '1-5', '6-10', '11+']) + f"  {q['tranco_pct']}%" + (f"   novelty {novtxt}" if novtxt else ''))

### TIBlender (D) population check

The same-input rows above are computed from `same_input_direct_records.jsonl`. This cell rebuilds the
TIBlender (D) population from `reports.jsonl` alone (Phishing/Fraud and Malware/C2 reports, IoC items
with `extraction_type` = Direct, attributed to the platforms recorded in `iocs.items[].platforms`) and
checks that it matches the evidence indicator-for-indicator and platform-for-platform. Novelty and
coverage depend only on this population; VirusTotal and Tranco values are evidence-file observations
and are not stored in the report records.


In [ ]:
# TIBlender (D) rows recomputed from reports.jsonl
from_reports = defaultdict(set); cats_by_key = defaultdict(set)
for r in reports:
    if r['paper_category'] in ('Phishing/Fraud', 'Malware/C2'):
        for i in r['iocs']['items']:
            if i['extraction_type'] == 'Direct':
                for pl in i['platforms']:
                    from_reports[pl].add((i['type'], i['value']))
                    cats_by_key[(pl, i['type'], i['value'])].add(r['paper_category'])
from_evidence = defaultdict(set)
for x in direct:
    from_evidence[x['platform']].add((x['type'], x['value']))
check = {}
for pl in PLATS:
    rep, ev = from_reports[pl], from_evidence[pl]
    dual = sum(len(cats_by_key[(pl, t, v)]) > 1 for t, v in rep)
    check[pl] = {'reports': len(rep), 'evidence': len(ev), 'reports_only': len(rep - ev), 'evidence_only': len(ev - rep),
                 'indicators_in_both_categories': dual}
    print(f"{pl:<9} reports {len(rep):>5}  evidence {len(ev):>5}  reports-only {len(rep - ev):>3}  evidence-only {len(ev - rep):>3}  in both categories {dual:>3}")
    assert rep == ev, pl
results['same_input_population_check'] = check
print('TIBlender (D) population from reports.jsonl matches the same-input evidence on every platform;')
print('indicators that occur in both a Phishing/Fraud and a Malware/C2 report are counted once, under the category recorded in the evidence file')


## 5. Public feed overlap, feed-scoped absence, early detection and lead time

In [ ]:
by_feed = defaultdict(list); tm = defaultdict(list)
for x in feeds:
    by_feed[x['feed']].append(x)
for x in timing:
    tm[x['feed']].append(x)
results['public_feeds'] = {}
print(f"{'Feed':<14}{'Denom.':>8}{'Overlap':>9}{'FSA':>8}{'Ahead':>10}{'Lead (h)':>10}")
for name in FEEDS:
    n = len(by_feed[name]); hit = sum(x['in_feed'] for x in by_feed[name])
    pairs = tm[name]; ahead = [x for x in pairs if x['detected_before_feed']]
    lead = round(sum(x['lead_hours'] for x in ahead) / len(ahead), 1) if ahead else None
    results['public_feeds'][name] = {'denominator': n, 'overlap': hit, 'FSA_pct': pct(n - hit, n), 'timed_pairs': len(pairs), 'ahead': len(ahead), 'ahead_pct': pct(len(ahead), len(pairs)), 'mean_lead_hours': lead}
    print(f"{name:<14}{n:>8,}{hit:>9,}{pct(n-hit, n):>7.1f}%{len(ahead):>4}/{len(pairs):<5}{pct(len(ahead), len(pairs)):>5.1f}%{lead:>10}")
tot_ahead = sum(x['detected_before_feed'] for x in timing)
print(f"All feeds: {tot_ahead}/{len(timing)} overlapping IoCs detected first ({pct(tot_ahead, len(timing))}%)")
kev = tm['CISA KEV']; kev_ahead = [x for x in kev if x['detected_before_feed']]
print(f"CISA KEV: {len(kev_ahead)}/{len(kev)} CVEs ahead, mean lead {round(sum(x['lead_hours'] for x in kev_ahead)/len(kev_ahead)):+} h, max lead {round(-min(x['lead_hours'] for x in kev_ahead)/24, 1)} days")
results['public_feeds']['all'] = {'ahead': tot_ahead, 'pairs': len(timing), 'ahead_pct': pct(tot_ahead, len(timing))}

## 6. Ablation study (per-report membership under each removed component)

In [ ]:
mem = defaultdict(Counter)
for x in ablation:
    mem[x['condition']][x['outcome']] += 1
skip_units = rows(EV / 'skip_cluster_units.jsonl')
results['ablation'] = {}
n = len(reports)
for cond, counts in mem.items():
    p = ablation_params[cond]
    results['ablation'][cond] = {'outcomes': dict(counts), 'outcome_pct': {k: pct(v, n) for k, v in counts.items()}, 'parameters': p}
    print(f"{cond}")
    for k, v in counts.items():
        print(f"   {k:<40} {v:>5,} ({pct(v, n)}%)")
    if cond == 'A2_without_adaptive_eval':
        lowq = sum(u['would_pass_without_adaptive_evaluation'] for u in skip_units)
        results['ablation'][cond]['low_quality_clusters_passing_without_gate'] = lowq
        print(f"   {'low-quality clusters passing without the gate':<40} {lowq:>5,} ({pct(lowq, n)}% of reports; report count would become {n + lowq:,})")
    print('   parameters:', json.dumps(p))

## 7. Manual review: quality audit (500 feed-overlap + 500 feed-absent reports) and false-negative reviews

In [ ]:
audit = [x for x in manual if x['review_set'] == 'quality_audit']
results['manual'] = {'quality_audit': {}, 'false_negative_review': {}}
print(f"{'Stratum':<14}{'n':>5}{'Report FP':>11}{'Unsupp. IoC':>13}{'Major issue':>13}")
fp_patterns = Counter(); major_patterns = Counter(); decisions = 0; disagreements = 0
for stratum in ['Feed-overlap', 'Feed-absent']:
    xs = [x for x in audit if x['stratum'] == stratum]
    fp = sum(x['report_level_false_positive'] for x in xs); un = sum(x['unsupported_ioc_issue'] for x in xs); mj = sum(x['major_quality_issue'] for x in xs)
    fp_patterns.update(x['false_positive_pattern'] for x in xs if x['false_positive_pattern']); major_patterns.update(x['major_quality_issue_pattern'] for x in xs if x['major_quality_issue_pattern'])
    decisions += 3 * len(xs); disagreements += sum(x['analyst_disagreements'] for x in xs)
    results['manual']['quality_audit'][stratum] = {'sample_n': len(xs), 'report_level_false_positive': fp, 'unsupported_ioc_issue': un, 'major_quality_issue': mj}
    print(f"{stratum:<14}{len(xs):>5}{fp:>6} ({pct(fp, len(xs))}%){un:>6} ({pct(un, len(xs))}%){mj:>6} ({pct(mj, len(xs))}%)")
print(f"Inter-annotator agreement: {decisions - disagreements}/{decisions} decisions ({pct(decisions - disagreements, decisions)}%)")
print('FP patterns:', dict(fp_patterns)); print('Major-issue patterns:', dict(major_patterns))
results['manual']['quality_audit']['agreement'] = {'decisions': decisions, 'agreed': decisions - disagreements, 'pct': pct(decisions - disagreements, decisions)}
results['manual']['quality_audit']['fp_patterns'] = dict(fp_patterns); results['manual']['quality_audit']['major_patterns'] = dict(major_patterns)
for stage in ['LOW clusters', 'Step 4 SKIP clusters']:
    xs = [x for x in manual if x['review_set'] == 'false_negative_review' and x['stage'] == stage]
    fn = [x for x in xs if x['final_label'] == 'false_negative_actionable_threat']; agree = sum(x['analyst_agreement'] for x in xs)
    results['manual']['false_negative_review'][stage] = {'sample_n': len(xs), 'false_negatives': len(fn), 'patterns': dict(Counter(x['false_negative_pattern'] for x in fn)), 'agreement': agree}
    print(f"FN review {stage:<22} n={len(xs)}  false negatives {len(fn)} ({pct(len(fn), len(xs))}%)  agreement {agree}/{len(xs)}  patterns {dict(Counter(x['false_negative_pattern'] for x in fn))}")

## 8. Record consistency: evaluation dates, KEV references, and audit strata

Recompute CVE counts and pre-listing report counts from report bodies; verify evidence references and feed-scoped manual strata.

In [ ]:
from datetime import timezone

def as_utc(value):
    if not value:
        return None
    parsed = datetime.fromisoformat(value.replace('Z', '+00:00'))
    return parsed.astimezone(timezone.utc) if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)

def report_cves(report):
    return {c.strip().lower() for c in report['report_text']['cves']}

def report_indicators(report):
    return {(i['type'], i['value'].strip().lower()) for i in report['iocs']['items']} | {('cve', c) for c in report_cves(report)}

period_start = as_utc('2026-03-16T00:00:00+00:00')
period_end = as_utc('2026-04-17T00:00:00+00:00')
missing_dates = sum(not r['created_at'] for r in reports)
outside_period = sum(not (period_start <= as_utc(r['created_at']) < period_end) for r in reports if r['created_at'])
vulnerability_reports = [r for r in reports if r['paper_category'] == 'Vulnerability']
body_cves = set().union(*(report_cves(r) for r in vulnerability_reports))
kev_evidence = [f for f in feeds if f['feed'] == 'CISA KEV']
kev_denominator = {f['value'].lower() for f in kev_evidence}
kev_hits = {f['value'].lower() for f in kev_evidence if f['in_feed']}
kev_dates = {t['value'].lower(): as_utc(t['feed_listing_time']) for t in timing if t['feed'] == 'CISA KEV'}
kev_reports = [r for r in vulnerability_reports if report_cves(r) & kev_hits]
pre_kev_reports = [r for r in kev_reports if any(as_utc(r['created_at']) < kev_dates[c] for c in report_cves(r) & kev_hits)]
early_cves = []
for c in sorted(kev_hits):
    referenced = [r for r in vulnerability_reports if c in report_cves(r)]
    detected = min(as_utc(r['created_at']) for r in referenced)
    if detected < kev_dates[c]:
        early_cves.append({'cve': c.upper(), 'first_detection': detected.isoformat(), 'kev_listed': kev_dates[c].isoformat(), 'lead_days': round((kev_dates[c] - detected).total_seconds() / 86400, 1), 'reports_before_listing': sum(as_utc(r['created_at']) < kev_dates[c] for r in referenced)})
feed_scopes = {'PhishTank': {'Phishing/Fraud'}, 'URLhaus': {'Malware/C2'}, 'MalwareBazaar': {'Malware/C2'}, 'CISA KEV': {'Vulnerability'}}
feed_hit_keys = defaultdict(set)
for f in feeds:
    if f['in_feed']:
        feed_hit_keys[f['feed']].add((f['type'], f['value'].strip().lower()))
report_by_id = {r['report_id']: r for r in reports}
strata_check = {}
for stratum in ['Feed-overlap', 'Feed-absent']:
    selected = [m for m in manual if m['review_set'] == 'quality_audit' and m['stratum'] == stratum]
    mismatches = []
    for m in selected:
        r = report_by_id[m['report_id']]
        matched = any(report_indicators(r) & ks for feed, ks in feed_hit_keys.items() if feed not in feed_scopes or r['paper_category'] in feed_scopes[feed])
        if matched != (stratum == 'Feed-overlap'):
            mismatches.append(m['report_id'])
    strata_check[stratum] = {'sample_n': len(selected), 'mismatches': mismatches}
timing_errors = []
for t in timing:
    r = report_by_id.get(t['report_id'])
    p = next((p for p in r['source_posts']['items'] if p['source_ref'] == t['source_ref']), None) if r else None
    valid = r is not None and p is not None
    if valid:
        valid = ((t['type'], t['value'].lower()) in report_indicators(r) and as_utc(t['detection_time']) == as_utc(r['created_at']) and t['source_post_time'] == p['timestamp'] and as_utc(t['source_post_time']) <= as_utc(t['detection_time']))
    if not valid:
        timing_errors.append({'feed': t['feed'], 'value': t['value'], 'report_id': t['report_id']})
results['record_consistency'] = {'reports_missing_created_at': missing_dates, 'reports_outside_period': outside_period, 'vulnerability_cves': len(body_cves), 'kev_denominator': len(kev_denominator), 'kev_cves_missing_from_reports': sorted(kev_denominator - body_cves), 'report_cves_missing_from_kev_denominator': sorted(body_cves - kev_denominator), 'kev_referencing_reports': len(kev_reports), 'reports_before_kev_listing': len(pre_kev_reports), 'early_kev_cves': early_cves, 'manual_strata': strata_check, 'timing_reference_errors': timing_errors}
print(json.dumps(results['record_consistency'], ensure_ascii=False, indent=2))


## 9. Save the recomputed figures

In [ ]:
Path('evaluation_figures.json').write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding='utf-8')
print('saved evaluation_figures.json')